In [ ]:
from rdkit import Chem
from rdkit.Chem import ChemicalForceFields
from rdkit.Chem import AllChem

# Create a molecule from a SMILES string
smiles = "C=O"  # Acetone
mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol)

# create a 3D conformer and optimize it
AllChem.EmbedMolecule(mol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(mol)

# Compute the MMFF properties of the molecule
mp_mmff = ChemicalForceFields.MMFFGetMoleculeProperties(mol)
mp_uff = ChemicalForceFields.UFFGetMoleculeForceField(mol)

# Iterate over each atom and print its MMFF atom type
for i in range(mol.GetNumAtoms()):
    atom = mol.GetAtomWithIdx(i).GetSymbol()
    atom_type_mmff = mp_mmff.GetMMFFAtomType(i)
    print(f"Atom {i + 1}: {atom} - MMFF Atom Type = {atom_type_mmff} - UFF Atom Type = {atom_type_uff}")

AttributeError: 'ForceField' object has no attribute 'GetAtomType'

# Riassunto

1) I flags di SDMolSupplier, riguardo removeHs e sanitize, si comportano come le funzioni di rdkit. Nota che il supplier ritorna una lista con la stessa lunghezza del file .sdf, ma con None dove la sanitizzazione è fallita.
2) RemoveHs di default fa anche sanitizzazione. Se non si esegue la sanitizzazione, allora non ci sono errori, ma appena si fa qualche operazione aggiuntiva sulla molecola, il palco cade e diversi errori vengono lanciati (es, calcolo force field)
3) RemoveHs dipende da stereochemistry
4) SanitizeMol è essenziale per evitare errori
5) Il kekulize non dovrebbe influenzare la rimozione degli idrogeni
6) Il kekulize è incluso nella funzione SanitizeMol
7) La stereochemistry non influenza la sanitizzazione
8) La stereochemistry non influenza la rimozione di Hs

9) Se io sanitizzo in lettura del file SDFMol e poi tolgo gli idrogeni senza sanetizzare, il calcolo force field da errore. Occorre sanetizzare nuovamente.

Nota che su QM9, se si sanitizza considerando la rimozione o meno degli idrogeni, si ottiene un numero diverso di molecole. Sorprendentmente, rimuovendo gli idrogeni, si riescono a salvare più molecole. L'ipotesi è che con gli idrogeni nella molecola, forse è più difficile far capire la molecola a RDKIT e quindi in più occasioni viene scartata.


In [19]:
from rdkit import Chem
from rdkit.Chem import ChemicalForceFields

filepath = "datasets/qm9/raw/gdb9.sdf"  # Replace with your SDF file path

suppl = Chem.SDMolSupplier(filepath, removeHs=False, sanitize=False)
suppl_false = Chem.SDMolSupplier(filepath, removeHs=False, sanitize=True)
suppl_true = Chem.SDMolSupplier(filepath, removeHs=True, sanitize=True)

suppl_filter = [mol for mol in suppl if mol is not None]
suppl_false_filter = [mol for mol in suppl_false if mol is not None]
suppl_true_filter = [mol for mol in suppl_true if mol is not None]

print("Molecules", len(suppl_filter))
print("Molecules with removeHs=False:", len(suppl_false_filter))
print("Molecules with removeHs=True:", len(suppl_true_filter))


OSError: File error: Bad input file datasets/qm9/raw/gdb9.sdf

In [20]:
# [Chem.SanitizeMol(i) for i in suppl_true_filter]
# [Chem.SanitizeMol(i) for i in suppl_false_filter]

suppl_false_filter_2nd =[Chem.RemoveHs(i, sanitize=True) for i in suppl_false_filter]
suppl_true_filter_2nd =[Chem.RemoveHs(i, sanitize=True) for i in suppl_true_filter]

print("Second round suppl_false_filter_2nd:", len(suppl_false_filter_2nd))
print("Second round suppl_true_filter_2nd:", len(suppl_true_filter_2nd))

Second round suppl_false_filter_2nd: 131970
Second round suppl_true_filter_2nd: 132737


In [22]:
# Cosa succede se io sanitizzo le molecole senza rimuovere gli idrogeni in lettura, poi rimuovo gli idrogeni e non sanifico?
suppl_false_filter_2nd =[Chem.RemoveHs(i, sanitize=False) for i in suppl_false_filter]
print("Second round suppl_false_filter_2nd:", len(suppl_false_filter_2nd))

Second round suppl_false_filter_2nd: 131970


In [18]:
import copy
suppl_filter_deepcopy = [copy.deepcopy(mol) for mol in suppl_filter]

suppl_filter_2nd = []

for i in suppl_filter_deepcopy:
    try:
        Chem.RemoveStereochemistry(i)
        # mol_no_h = Chem.SanitizeMol(i)
        mol_no_h = Chem.RemoveHs(i, sanitize=True)
        suppl_filter_2nd.append(mol_no_h)
    except:
        continue

print("Second round suppl_filter_2nd:", len(suppl_filter_2nd))

[17:06:13] Explicit valence for atom # 1 C, 5, is greater than permitted
[17:06:13] Explicit valence for atom # 1 C, 5, is greater than permitted
[17:06:13] Explicit valence for atom # 4 C, 5, is greater than permitted
[17:06:13] Explicit valence for atom # 2 C, 5, is greater than permitted
[17:06:13] Explicit valence for atom # 2 C, 5, is greater than permitted
[17:06:13] Explicit valence for atom # 2 C, 5, is greater than permitted
[17:06:13] Explicit valence for atom # 2 C, 5, is greater than permitted
[17:06:13] Explicit valence for atom # 2 C, 5, is greater than permitted
[17:06:13] Explicit valence for atom # 1 C, 5, is greater than permitted
[17:06:13] Explicit valence for atom # 1 C, 5, is greater than permitted
[17:06:13] Explicit valence for atom # 1 C, 4, is greater than permitted
[17:06:13] Explicit valence for atom # 1 C, 4, is greater than permitted
[17:06:13] Explicit valence for atom # 1 C, 4, is greater than permitted
[17:06:13] Explicit valence for atom # 1 C, 4, is g

Second round suppl_filter_2nd: 132737


[17:06:18] Explicit valence for atom # 7 C, 5, is greater than permitted
[17:06:18] Explicit valence for atom # 6 C, 5, is greater than permitted
[17:06:18] Explicit valence for atom # 7 C, 5, is greater than permitted
